In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import GridSearchCV
import re
from nltk.stem import WordNetLemmatizer

#Carga de datos
train_data = pd.read_csv("../../Data/train_indexado.csv")
test_data = pd.read_csv("../../Data/test_indexado.csv")

# Definir las clases de emociones
emotion_classes = train_data.columns[2:].tolist()

# --- PREPROCESAMIENTO PARA LEMATIZACIÓN ---
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    words = re.findall(r'\b\w+\b', text.lower())
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)

X_train_lem = train_data['Text'].apply(preprocess_text)
X_test_lem = test_data['Text'].apply(preprocess_text)

# TF-IDF VECTORIZACIÓN
vectorizer = TfidfVectorizer(lowercase=True, strip_accents="unicode", max_features=10000)
X_train = vectorizer.fit_transform(X_train_lem)
X_test = vectorizer.transform(X_test_lem)
y_train = np.asarray(train_data[emotion_classes])
y_test = np.asarray(test_data[emotion_classes])

# Selección de features con Chi2 = 1000
selector = SelectKBest(score_func=chi2, k=1000)
X_train_chi = selector.fit_transform(X_train, y_train)
X_test_chi = selector.transform(X_test)

print(f"Datos preparados: {X_train_chi.shape[0]} muestras de entrenamiento, {X_train_chi.shape[1]} features")

Datos preparados: 43410 muestras de entrenamiento, 1000 features


In [2]:
# Hiperparametrización de SVM con Chi2=1000
print("Iniciando búsqueda de hiperparámetros para SVM (Chi2=1000)...")

# Parámetros a probar
param_grid = {
    'estimator__C': [0.1, 1.0, 10.0, 100.0],
    'estimator__kernel': ['linear'],
    'estimator__gamma': ['scale', 'auto']
}

# Crear el modelo base
svm_base = SVC(random_state=42)
multi_svm = MultiOutputClassifier(svm_base, n_jobs=1)




# GridSearchCV
grid_search = GridSearchCV(
    estimator=multi_svm,
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

print(f"Total de combinaciones a probar: {len(param_grid['estimator__C']) * len(param_grid['estimator__kernel']) * len(param_grid['estimator__gamma'])}")

# Entrenar
grid_search.fit(X_train_chi, y_train)

print(f"\nMejores parámetros: {grid_search.best_params_}")
print(f"Mejor score CV (F1 macro): {grid_search.best_score_:.5f}")

Iniciando búsqueda de hiperparámetros para SVM (Chi2=1000)...
Total de combinaciones a probar: 8
Fitting 3 folds for each of 8 candidates, totalling 24 fits

Mejores parámetros: {'estimator__C': 100.0, 'estimator__gamma': 'scale', 'estimator__kernel': 'linear'}
Mejor score CV (F1 macro): 0.36322


In [3]:
# Evaluar el mejor modelo
best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test_chi)

# Métricas del mejor modelo
report_dict_best = classification_report(y_test, y_pred_best, output_dict=True, zero_division=0)
f1_macro_best = report_dict_best["macro avg"]["f1-score"]
recall_macro_best = report_dict_best["macro avg"]["recall"]

print("\n=== RESULTADOS DEL MEJOR SVM (Chi2=1000) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_best):.5f}")
print(f"F1 Score (macro avg): {f1_macro_best:.5f}")
print(f"Recall Score (macro avg): {recall_macro_best:.5f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred_best, zero_division=0)}")

# Comparar con SVM por defecto (Chi2=1000)
svm_default = SVC(kernel='linear', random_state=42, max_iter=1000)
multi_svm_default = MultiOutputClassifier(svm_default)
multi_svm_default.fit(X_train_chi, y_train)
y_pred_default = multi_svm_default.predict(X_test_chi)

report_dict_default = classification_report(y_test, y_pred_default, output_dict=True, zero_division=0)
f1_macro_default = report_dict_default["macro avg"]["f1-score"]
recall_macro_default = report_dict_default["macro avg"]["recall"]

print("\n=== COMPARACIÓN CON SVM POR DEFECTO (Chi2=1000) ===")
print(f"Accuracy por defecto: {accuracy_score(y_test, y_pred_default):.5f}")
print(f"F1 Score por defecto (macro avg): {f1_macro_default:.5f}")
print(f"Recall Score por defecto (macro avg): {recall_macro_default:.5f}")

print("\n=== MEJORA OBTENIDA ===")
print(f"Mejora en Accuracy: {accuracy_score(y_test, y_pred_best) - accuracy_score(y_test, y_pred_default):.5f}")
print(f"Mejora en F1 Score: {f1_macro_best - f1_macro_default:.5f}")
print(f"Mejora en Recall: {recall_macro_best - recall_macro_default:.5f}")

# Mostrar configuración óptima
print("\n=== CONFIGURACIÓN ÓPTIMA ===")
print(f"Chi2 features: 1000")
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")


=== RESULTADOS DEL MEJOR SVM (Chi2=1000) ===
Accuracy: 0.40962
F1 Score (macro avg): 0.36104
Recall Score (macro avg): 0.30792

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.47      0.57       504
           1       0.78      0.78      0.78       264
           2       0.59      0.19      0.28       198
           3       0.69      0.06      0.10       320
           4       0.69      0.10      0.17       351
           5       0.54      0.05      0.09       135
           6       0.64      0.09      0.16       153
           7       0.80      0.04      0.08       284
           8       0.50      0.24      0.33        83
           9       0.47      0.05      0.10       151
          10       0.36      0.02      0.04       267
          11       0.67      0.25      0.37       123
          12       0.42      0.30      0.35        37
          13       0.68      0.22      0.34       103
          14       0.76      0.53    

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/ec2-user/anaconda3/env


=== COMPARACIÓN CON SVM POR DEFECTO (Chi2=1000) ===
Accuracy por defecto: 0.01142
F1 Score por defecto (macro avg): 0.24913
Recall Score por defecto (macro avg): 0.48521

=== MEJORA OBTENIDA ===
Mejora en Accuracy: 0.39819
Mejora en F1 Score: 0.11191
Mejora en Recall: -0.17729

=== CONFIGURACIÓN ÓPTIMA ===
Chi2 features: 1000
estimator__C: 100.0
estimator__gamma: scale
estimator__kernel: linear
